# LAB 09 — Fine-tuning DistilBERT cho NER (CoNLL-2003)
Notebook này gồm: pipeline token classification từng bước (core), đường cong huấn luyện, và Experiment 2–6 + Context analysis.
Huấn luyện chạy bằng `python train_ner.py` (2 epoch, CPU); notebook đọc checkpoint `ner_checkpoint/`, `training_log.json`, `transformer_predictions.json`
và `baseline_predictions.json` (từ `baseline_ner.py`). **Cả hai model dùng đúng cùng một test set (3,453 câu, 5,648 entity).**

In [1]:
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from ner_utils import *
from ner_model import load_model, encode_sentence, predict, MODEL_NAME
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 120)
ds = load_conll()
model, tok = load_model()
test = ds["test"]
words_te = [ex["tokens"] for ex in test]
gold = [tags_to_labels(ex["ner_tags"]) for ex in test]
_tf = json.load(open("transformer_predictions.json"))
pred_tf, conf_tf = _tf["pred"], _tf["conf"]
pred_bl = json.load(open("baseline_predictions.json"))["pred"]
log = json.load(open("training_log.json"))
print(MODEL_NAME, "| params:", f"{log['n_params']:,}", "| test sentences:", len(test))

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 4613.35it/s]

distilbert-base-cased | params: 65,197,833 | test sentences: 3453


## 1. Core — pipeline token classification từng bước (không dùng `pipeline("ner")`)
`words → tokenizer(is_split_into_words=True) → input_ids / attention_mask / word_ids → Transformer → logits (T×9) → argmax → nhãn theo word`

In [2]:
words = ["Barack", "Obama", "visited", "Reykjavik", "with", "Nikolaus", "van", "der", "Pas", "."]
word_tags = ["B-PER", "I-PER", "O", "B-LOC", "O", "B-PER", "I-PER", "I-PER", "I-PER", "O"]
enc = tok(words, is_split_into_words=True, truncation=True, padding=True, return_tensors="pt")
wids = enc.word_ids()
labels = align_labels_first_subtoken(wids, [LABEL2ID[t] for t in word_tags])
with torch.no_grad():
    logits = model(**enc).logits
probs = logits.softmax(-1)[0]
pred_ids = probs.argmax(-1)
df = pd.DataFrame({"token": tok.convert_ids_to_tokens(enc["input_ids"][0]), "input_id": enc["input_ids"][0].tolist(),
                   "attn_mask": enc["attention_mask"][0].tolist(), "word_id": wids,
                   "label": [ID2LABEL[l] if l != -100 else "IGNORE" for l in labels],
                   "pred": [ID2LABEL[int(p)] for p in pred_ids], "p(pred)": probs.max(-1).values.numpy().round(3)})
print("input_ids:", tuple(enc["input_ids"].shape), "| logits:", tuple(logits.shape), "= (batch, tokens, 9 nhãn)")
print(df.to_string())
wp, wc = predict(model, tok, [encode_sentence(tok, words)], [len(words)])
print("\nDự đoán theo word:", list(zip(words, wp[0])))
print("Entity:", [(t, " ".join(words[s:e])) for t, s, e in sorted(spans_from_tags(wp[0]), key=lambda x: x[1])])

input_ids: (1, 17) | logits: (1, 17, 9) = (batch, tokens, 9 nhãn)
      token  input_id  attn_mask  word_id   label   pred  p(pred)
0     [CLS]       101          1      NaN  IGNORE      O    0.913
1    Barack     14319          1      0.0   B-PER  B-PER    0.991
2     Obama      7661          1      1.0   I-PER  I-PER    0.984
3   visited      3891          1      2.0       O      O    1.000
4       Rey     12952          1      3.0   B-LOC  B-LOC    0.870
5       ##k      1377          1      3.0  IGNORE  I-LOC    0.742
6      ##ja      3174          1      3.0  IGNORE  I-LOC    0.555
7     ##vik     13160          1      3.0  IGNORE  I-LOC    0.352
8      with      1114          1      4.0       O      O    1.000
9    Nikola     28010          1      5.0   B-PER  B-PER    0.997
10     ##us      1361          1      5.0  IGNORE  I-PER    0.907
11      van      3498          1      6.0   I-PER  I-PER    0.998
12      der      4167          1      7.0   I-PER  I-PER    0.993
13       P

## 2. Huấn luyện
`DistilBERT-base-cased` (66M tham số) + `Linear(768→9)`; AdamW lr 5e-5, weight decay 0.01, batch 16, 2 epoch, warmup 10% + linear decay, clip 1.0, seed 42,
`CrossEntropyLoss(ignore_index=-100)`; chọn epoch có validation entity-F1 cao nhất (checkpoint lưu tại `ner_checkpoint/`). Chạy trên **CPU**.

In [3]:
L = pd.DataFrame(log["log"]); print(L.round(4).to_string(index=False)); print("train time: %.0fs" % log["train_time_s"])
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].plot(L.epoch, L.train_loss, "o-"); ax[0].set_title("Train loss"); ax[0].set_xlabel("epoch")
ax[1].plot(L.epoch, L.val_f1, "o-", color="#e45756"); ax[1].set_title("Validation entity-level F1"); ax[1].set_xlabel("epoch")
for a in ax: a.set_xticks(L.epoch)
plt.tight_layout(); plt.savefig("training_curves.png", dpi=130); plt.show()

 epoch  train_loss  val_f1  elapsed_s
     1      0.1797  0.9157  1859.6356
     2      0.0257  0.9351  3216.6679
train time: 3217s


## 3. Experiment 2 + 3 — Baseline vs Transformer, entity-level vs token-level
Metric chính là **entity-level** (exact match: đúng cả type lẫn span). Token accuracy chỉ in để so sánh.
Đối chiếu cài đặt tự viết với `seqeval` để chắc chắn.

In [4]:
from seqeval.metrics import precision_score, recall_score, f1_score
bl = json.load(open("baseline_metrics.json"))
res = {}
for name, pred in [("Baseline (TF-IDF-style features + LogReg)", pred_bl), ("DistilBERT-cased (fine-tuned)", pred_tf)]:
    m = entity_metrics(gold, pred); res[name] = m
    sq = (precision_score(gold, pred), recall_score(gold, pred), f1_score(gold, pred))
    o = m["overall"]
    assert abs(o["f1"] - sq[2]) < 1e-9 and abs(o["precision"] - sq[0]) < 1e-9 and abs(o["recall"] - sq[1]) < 1e-9
    print(f"{name}: entity P/R/F1 = {o['precision']:.4f}/{o['recall']:.4f}/{o['f1']:.4f}  (seqeval khớp)")
names = list(res)
rows = []
for (name, m), npar, tt in zip(res.items(), [9 * bl["n_features"] + 9, log["n_params"]], [bl["train_time_s"], log["train_time_s"]]):
    o = m["overall"]; pred = pred_bl if name.startswith("Baseline") else pred_tf
    tp_, tr_, tf_ = token_level_prf(gold, pred)
    rows.append(dict(model=name, precision=o["precision"], recall=o["recall"], f1=o["f1"], tp=o["tp"], fp=o["fp"], fn=o["fn"],
                     token_accuracy=token_accuracy(gold, pred), token_f1_nonO=tf_, n_params=npar, train_time_s=round(tt, 1)))
R = pd.DataFrame(rows); R.to_csv("results.csv", index=False)
print(R.round(4).to_string(index=False))

Baseline (TF-IDF-style features + LogReg): entity P/R/F1 = 0.7405/0.7916/0.7652  (seqeval khớp)


DistilBERT-cased (fine-tuned): entity P/R/F1 = 0.8826/0.9026/0.8925  (seqeval khớp)
                                    model  precision  recall     f1   tp   fp   fn  token_accuracy  token_f1_nonO  n_params  train_time_s
Baseline (TF-IDF-style features + LogReg)     0.7405  0.7916 0.7652 4471 1567 1177          0.9605         0.8219    996705          31.3
            DistilBERT-cased (fine-tuned)     0.8826  0.9026 0.8925 5098  678  550          0.9788         0.9090  65197833        3216.9


In [5]:
# Token-level vs entity-level: cùng dự đoán, hai kết luận khác nhau
print("Baseline    : token accuracy %.4f  vs entity F1 %.4f  (gap %.3f)" % (R.token_accuracy[0], R.f1[0], R.token_accuracy[0] - R.f1[0]))
print("Transformer : token accuracy %.4f  vs entity F1 %.4f  (gap %.3f)" % (R.token_accuracy[1], R.f1[1], R.token_accuracy[1] - R.f1[1]))
pO = np.mean([t == "O" for g in gold for t in g])
print("Tỷ lệ nhãn 'O' trong test: %.3f -> luôn dự đoán 'O' cho token accuracy %.3f nhưng entity F1 = 0" % (pO, pO))
g_, p_ = [["B-LOC", "I-LOC"]], [["B-LOC", "O"]]
print("\nVí dụ 'New York': token accuracy = %.2f ; entity P/R/F1 = %s" % (token_accuracy(g_, p_), [round(entity_metrics(g_, p_)['overall'][k], 2) for k in ("precision", "recall", "f1")]))
def invalid_bio(seqs):
    n = 0
    for s in seqs:
        prev = "O"
        for t in s:
            n += t.startswith("I-") and prev[2:] != t[2:]; prev = t
    return n
print("\n# chuyển tiếp BIO không hợp lệ (I-X không đứng sau B-X/I-X): baseline = %d, transformer = %d" % (invalid_bio(pred_bl), invalid_bio(pred_tf)))

Baseline    : token accuracy 0.9605  vs entity F1 0.7652  (gap 0.195)
Transformer : token accuracy 0.9788  vs entity F1 0.8925  (gap 0.086)
Tỷ lệ nhãn 'O' trong test: 0.825 -> luôn dự đoán 'O' cho token accuracy 0.825 nhưng entity F1 = 0

Ví dụ 'New York': token accuracy = 0.50 ; entity P/R/F1 = [0.0, 0.0, 0.0]

# chuyển tiếp BIO không hợp lệ (I-X không đứng sau B-X/I-X): baseline = 462, transformer = 72


## 4. Experiment 4 — Entity type analysis (`entity_metrics.csv`)

In [6]:
rows = []
for name, m in res.items():
    for t in ENTITY_TYPES + ["OVERALL"]:
        r = m["overall"] if t == "OVERALL" else m["per_type"][t]
        rows.append(dict(model=name, entity=t, precision=r["precision"], recall=r["recall"], f1=r["f1"], tp=r["tp"], fp=r["fp"], fn=r["fn"], support=r["support"]))
E = pd.DataFrame(rows); E.to_csv("entity_metrics.csv", index=False)
print(E.round(4).to_string(index=False))
ent_tr = Counter(t for ex in ds["train"] for (t, _, _) in spans_from_tags(tags_to_labels(ex["ner_tags"])))
print("\n# entity trong train:", dict(ent_tr))

                                    model  entity  precision  recall     f1   tp   fp   fn  support
Baseline (TF-IDF-style features + LogReg)     PER     0.7697  0.8330 0.8001 1347  403  270     1617
Baseline (TF-IDF-style features + LogReg)     ORG     0.6407  0.6924 0.6655 1150  645  511     1661
Baseline (TF-IDF-style features + LogReg)     LOC     0.8160  0.8585 0.8367 1432  323  236     1668
Baseline (TF-IDF-style features + LogReg)    MISC     0.7344  0.7721 0.7528  542  196  160      702
Baseline (TF-IDF-style features + LogReg) OVERALL     0.7405  0.7916 0.7652 4471 1567 1177     5648
            DistilBERT-cased (fine-tuned)     PER     0.9438  0.9549 0.9493 1544   92   73     1617
            DistilBERT-cased (fine-tuned)     ORG     0.8384  0.8838 0.8605 1468  283  193     1661
            DistilBERT-cased (fine-tuned)     LOC     0.9194  0.9095 0.9144 1517  133  151     1668
            DistilBERT-cased (fine-tuned)    MISC     0.7700  0.8105 0.7897  569  170  133      702



# entity trong train: {'MISC': 3438, 'ORG': 6321, 'PER': 6600, 'LOC': 7140}


## 5. Experiment 5 — Entity length (`entity_length_analysis.png`)

In [7]:
rows = []
for name, m in res.items():
    for Lg, r in m["per_length"].items():
        rows.append(dict(model=name, length=Lg, precision=r["precision"], recall=r["recall"], f1=r["f1"], support=r["support"], tp=r["tp"], fp=r["fp"], fn=r["fn"]))
LEN = pd.DataFrame(rows); LEN.to_csv("entity_length_analysis.csv", index=False)
print(LEN.round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(7.5, 4))
x = np.arange(4); w = 0.38
for k, n in enumerate(names):
    f = [res[n]["per_length"][l]["f1"] for l in ("1", "2", "3", "4+")]
    ax.bar(x + (k - 0.5) * w, f, w, label=n, color=["#9ecae9", "#e45756"][k])
    for xi, fi in zip(x + (k - 0.5) * w, f): ax.text(xi, fi + 0.01, f"{fi:.2f}", ha="center", fontsize=8)
sup = [res[names[1]]["per_length"][l]["support"] for l in ("1", "2", "3", "4+")]
ax.set_xticks(x); ax.set_xticklabels([f"{l} token\n(n={s})" for l, s in zip(("1", "2", "3", "4+"), sup)])
ax.set_ylim(0, 1.1); ax.set_ylabel("entity-level F1"); ax.set_title("F1 theo độ dài entity (test set)"); ax.legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.savefig("entity_length_analysis.png", dpi=130); plt.show()

                                    model length  precision  recall     f1  support   tp   fp  fn
Baseline (TF-IDF-style features + LogReg)      1     0.6882  0.8114 0.7447     3574 2900 1314 674
Baseline (TF-IDF-style features + LogReg)      2     0.8684  0.7990 0.8323     1776 1419  215 357
Baseline (TF-IDF-style features + LogReg)      3     0.8258  0.5424 0.6547      236  128   27 108
Baseline (TF-IDF-style features + LogReg)     4+     0.6857  0.3871 0.4948       62   24   11  38
            DistilBERT-cased (fine-tuned)      1     0.8726  0.9046 0.8883     3574 3233  472 341
            DistilBERT-cased (fine-tuned)      2     0.9133  0.9071 0.9102     1776 1611  153 165
            DistilBERT-cased (fine-tuned)      3     0.8475  0.8475 0.8475      236  200   36  36
            DistilBERT-cased (fine-tuned)     4+     0.7606  0.8710 0.8120       62   54   17   8


### Entity hiếm / chưa thấy trong training (kiểm chứng Prediction 3)

In [8]:
train_surfaces = Counter()
for ex in ds["train"]:
    for t, s, e in spans_from_tags(tags_to_labels(ex["ner_tags"])):
        train_surfaces[" ".join(ex["tokens"][s:e]).lower()] += 1
def seen_bucket(surface):
    c = train_surfaces.get(surface.lower(), 0)
    return "unseen (0)" if c == 0 else ("rare (1-4)" if c < 5 else "frequent (5+)")
rows = []
for name, pred in [("Baseline", pred_bl), ("Transformer", pred_tf)]:
    cnt = defaultdict(lambda: [0, 0, 0])  # tp, fp, fn
    for w, g, p in zip(words_te, gold, pred):
        gs, ps = spans_from_tags(g), spans_from_tags(p)
        for e in ps & gs: cnt[seen_bucket(" ".join(w[e[1]:e[2]]))][0] += 1
        for e in ps - gs: cnt[seen_bucket(" ".join(w[e[1]:e[2]]))][1] += 1
        for e in gs - ps: cnt[seen_bucket(" ".join(w[e[1]:e[2]]))][2] += 1
    for b, (tp, fp, fn) in cnt.items():
        P = tp / (tp + fp) if tp + fp else 0; Rr = tp / (tp + fn) if tp + fn else 0
        rows.append(dict(model=name, bucket=b, gold_entities=tp + fn, tp=tp, fp=fp, fn=fn, precision=P, recall=Rr, f1=2 * P * Rr / (P + Rr) if P + Rr else 0))
S = pd.DataFrame(rows).sort_values(["bucket", "model"]); print(S.round(4).to_string(index=False)); S.to_csv("seen_unseen_analysis.csv", index=False)

      model        bucket  gold_entities   tp   fp  fn  precision  recall     f1
   Baseline frequent (5+)           2209 2083  226 126     0.9021  0.9430 0.9221
Transformer frequent (5+)           2209 2088  124 121     0.9439  0.9452 0.9446
   Baseline    rare (1-4)            939  740  198 199     0.7889  0.7881 0.7885
Transformer    rare (1-4)            939  828   75 111     0.9169  0.8818 0.8990
   Baseline    unseen (0)           2500 1648 1143 852     0.5905  0.6592 0.6229
Transformer    unseen (0)           2500 2182  479 318     0.8200  0.8728 0.8456


## 6. Confusion matrix (mức word, nhãn BIO 9 lớp) — `confusion_matrix.png`

In [9]:
def conf_mat(pred):
    M = np.zeros((9, 9), dtype=int)
    for g, p in zip(gold, pred):
        for a, b in zip(g, p): M[LABEL2ID[a], LABEL2ID[b]] += 1
    return M
fig, axs = plt.subplots(1, 2, figsize=(15, 6))
for ax, (name, pred) in zip(axs, [("Baseline", pred_bl), ("Transformer (DistilBERT)", pred_tf)]):
    M = conf_mat(pred); Mn = M / M.sum(1, keepdims=True)
    ax.imshow(Mn, cmap="Blues", vmin=0, vmax=1)
    for i in range(9):
        for j in range(9):
            if M[i, j]: ax.text(j, i, M[i, j], ha="center", va="center", fontsize=7, color="white" if Mn[i, j] > 0.5 else "black")
    ax.set_xticks(range(9)); ax.set_xticklabels(LABELS, rotation=45); ax.set_yticks(range(9)); ax.set_yticklabels(LABELS)
    ax.set_xlabel("Dự đoán"); ax.set_ylabel("Gold"); ax.set_title(f"{name} — số word (màu = tỷ lệ theo hàng)")
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=130); plt.show()
M = conf_mat(pred_tf); off = M.copy(); np.fill_diagonal(off, 0)
top = np.dstack(np.unravel_index(np.argsort(-off.ravel())[:8], off.shape))[0]
print("Top nhầm lẫn (Transformer) gold->pred:", [(ID2LABEL[i], ID2LABEL[j], int(off[i, j])) for i, j in top])

Top nhầm lẫn (Transformer) gold->pred: [('O', 'B-ORG', 75), ('B-LOC', 'B-ORG', 69), ('O', 'I-ORG', 58), ('O', 'I-MISC', 57), ('O', 'B-MISC', 51), ('B-ORG', 'B-LOC', 48), ('B-ORG', 'O', 40), ('B-MISC', 'B-ORG', 40)]


## 7. Experiment 6 — Error analysis (Transformer)
Phân loại theo entity: **false positive** (entity dự đoán không chồng lấn entity gold nào), **false negative** (entity gold bị bỏ sót hoàn toàn), **boundary error**
(cùng type, chồng lấn nhưng span khác) và **wrong type** (chồng lấn nhưng khác type). Lấy ngẫu nhiên (seed 42) 6 mẫu mỗi loại; vào `error_samples.csv`.

In [10]:
errs = error_lists(words_te, gold, pred_tf)
print({k: len(v) for k, v in errs.items()}, "| tổng entity gold:", res[names[1]]["overall"]["support"])
rng = np.random.RandomState(SEED)
rows = []
def hi(w, sp): return "[" + " ".join(w[sp[1]:sp[2]]) + f"]({sp[0]})" if sp else "-"
for kind in ["fp", "fn", "boundary", "wrong_type"]:
    items = errs[kind]; pick = sorted(rng.choice(len(items), min(6, len(items)), replace=False))
    for k in pick:
        idx, pe, ge = items[k]
        w = words_te[idx]
        rows.append(dict(kind=kind, sent_id=idx, text=" ".join(w)[:240], gold=hi(w, ge), prediction=hi(w, pe),
                         pred_conf=round(conf_tf[idx][pe[1]], 3) if pe else None))
ES = pd.DataFrame(rows); ES.to_csv("error_samples.csv", index=False)
for r in ES.itertuples():
    print(f"[{r.kind:<10}] #{r.sent_id:<5} gold={r.gold:<34} pred={r.prediction:<34} conf={r.pred_conf}\n             {r.text}")

{'fp': 199, 'fn': 102, 'boundary': 118, 'wrong_type': 361} | tổng entity gold: 5648
[fp        ] #416   gold=-                                  pred=[AFC](ORG)                         conf=0.905
             The injury-plagued Indianapolis Colts lost another quarterback on Thursday but last year 's AFC finalists rallied together to shoot down the Philadelphia Eagles 37-10 in a showdown of playoff contenders .
[fp        ] #1607  gold=-                                  pred=[Feedlots](ORG)                    conf=0.787
             Feedlots reporting moderate inquiry .
[fp        ] #1611  gold=-                                  pred=[Choice](ORG)                      conf=0.967
             Slaughter Heifers - Select and Choice 2-3 1050-1100 lbs 67.00 .
[fp        ] #1713  gold=-                                  pred=[Oats](ORG)                        conf=0.864
             Oats 286.5 284.6 31.9 937.3 581.2
[fp        ] #2125  gold=-                                  pred=[Mantua](LOC) 

In [11]:
cnt = Counter()
for kind in ["fp", "fn"]:
    for idx, pe, ge in errs[kind]: cnt[(kind, (pe or ge)[0])] += 1
for kind in ["boundary", "wrong_type"]:
    for idx, pe, ge in errs[kind]: cnt[(kind, ge[0])] += 1
print(pd.Series(cnt).unstack().fillna(0).astype(int))
print("\nwrong_type gold -> pred:", Counter((g[0], p[0]) for _, p, g in errs["wrong_type"]).most_common(8))
ok, bad = [], []
for i, (g, p) in enumerate(zip(gold, pred_tf)):
    gs = spans_from_tags(g)
    for sp in spans_from_tags(p): (ok if sp in gs else bad).append(conf_tf[i][sp[1]])
print("Độ tin cậy trung bình tại token đầu của entity dự đoán: đúng = %.3f ; sai = %.3f" % (np.mean(ok), np.mean(bad)))

            LOC  MISC  ORG  PER
boundary     15    41   53    9
fn           18    26   37   21
fp           29    60   81   29
wrong_type  124    73  121   43

wrong_type gold -> pred: [(('LOC', 'ORG'), 76), (('ORG', 'LOC'), 58), (('MISC', 'ORG'), 44), (('LOC', 'MISC'), 34), (('ORG', 'MISC'), 32), (('ORG', 'PER'), 31), (('PER', 'ORG'), 29), (('MISC', 'LOC'), 20)]
Độ tin cậy trung bình tại token đầu của entity dự đoán: đúng = 0.979 ; sai = 0.805


## 8. Context analysis — cùng surface form, khác entity type

In [12]:
occ = defaultdict(list)
for i, (w, g) in enumerate(zip(words_te, gold)):
    for t, s, e in spans_from_tags(g): occ[" ".join(w[s:e])].append((i, t, s, e))
multi = {k: v for k, v in occ.items() if len({x[1] for x in v}) > 1 and min(Counter(x[1] for x in v).values()) >= 2}
print("Surface form có >=2 type khác nhau trong gold (mỗi type >=2 lần):", {k: dict(Counter(x[1] for x in v)) for k, v in multi.items()})
def pred_type(pred, i, s, e):
    for t, ss, ee in spans_from_tags(pred[i]):
        if ss == s and ee == e: return t
    return "-"
for sf in sorted(multi, key=lambda k: -len(multi[k]))[:5]:
    print("\n==", sf)
    shown = Counter()
    for i, t, s, e in multi[sf]:
        if shown[t] >= 3: continue
        shown[t] += 1
        ctx = " ".join(words_te[i][max(0, s - 5): e + 6])
        print(f"  gold={t:<5} baseline={pred_type(pred_bl, i, s, e):<5} transformer={pred_type(pred_tf, i, s, e):<5} | ...{ctx}...")
acc = defaultdict(lambda: [0, 0, 0])
for sf, v in multi.items():
    for i, t, s, e in v:
        acc[sf][0] += 1; acc[sf][1] += pred_type(pred_bl, i, s, e) == t; acc[sf][2] += pred_type(pred_tf, i, s, e) == t
print("\nSố lần đoán đúng (span + type): (baseline, transformer, tổng) =", {k: (v[1], v[2], v[0]) for k, v in acc.items()})
tot = np.array(list(acc.values())).sum(0); print("Tổng:", tot[0], "occurrence; baseline đúng", tot[1], "(%.1f%%); transformer đúng" % (100 * tot[1] / tot[0]), tot[2], "(%.1f%%)" % (100 * tot[2] / tot[0]))

Surface form có >=2 type khác nhau trong gold (mỗi type >=2 lần): {'Barcelona': {'LOC': 2, 'ORG': 22}, 'Victoria': {'LOC': 4, 'ORG': 3}, 'Tasmania': {'LOC': 2, 'ORG': 3}, 'NEW YORK': {'LOC': 19, 'ORG': 3}, 'League': {'ORG': 6, 'LOC': 2}, 'HARTFORD': {'ORG': 2, 'LOC': 2}, 'BOSTON': {'ORG': 6, 'LOC': 2}, 'PITTSBURGH': {'ORG': 4, 'LOC': 2}, 'NEW JERSEY': {'ORG': 6, 'LOC': 2}, 'WASHINGTON': {'ORG': 8, 'LOC': 4}, 'DETROIT': {'ORG': 7, 'LOC': 2}, 'DALLAS': {'ORG': 6, 'LOC': 3}, 'CHICAGO': {'ORG': 7, 'LOC': 6}, 'COLORADO': {'ORG': 3, 'LOC': 2}, 'EDMONTON': {'ORG': 2, 'LOC': 2}, 'Philadelphia': {'LOC': 3, 'ORG': 2}, 'MIAMI': {'ORG': 5, 'LOC': 2}, 'HOUSTON': {'ORG': 3, 'LOC': 3}, 'SEATTLE': {'ORG': 3, 'LOC': 2}, 'American': {'MISC': 6, 'ORG': 4}, 'Santa Fe': {'LOC': 2, 'ORG': 16}, 'Busang': {'ORG': 4, 'LOC': 5}, 'Wimbledon': {'LOC': 4, 'ORG': 4}, 'Charlton': {'ORG': 2, 'PER': 4}}

== Barcelona
  gold=LOC   baseline=LOC   transformer=LOC   | ...status committee , meeting in Barcelona , decided t

In [13]:
demo = ["Washington signed the treaty with Moscow .".split(), "He lives in Washington , near Seattle .".split(),
        "Washington beat Dallas 24-17 on Sunday .".split(), "Jordan scored 30 points for Chicago .".split(), "The trip to Jordan was long .".split(),
        "Leeds won 2-0 at Everton .".split(), "She was born in Leeds .".split()]
for w in demo:
    p, c = predict(model, tok, [encode_sentence(tok, w)], [len(w)])
    print(" ".join(w), "->", [(" ".join(w[s:e]), t) for t, s, e in sorted(spans_from_tags(p[0]), key=lambda x: x[1])])

Washington signed the treaty with Moscow . -> [('Washington', 'LOC'), ('Moscow', 'LOC')]
He lives in Washington , near Seattle . -> [('Washington', 'LOC'), ('Seattle', 'LOC')]
Washington beat Dallas 24-17 on Sunday . -> [('Washington', 'ORG'), ('Dallas', 'ORG')]


Jordan scored 30 points for Chicago . -> [('Jordan', 'PER'), ('Chicago', 'LOC')]
The trip to Jordan was long . -> [('Jordan', 'LOC')]
Leeds won 2-0 at Everton . -> [('Leeds', 'ORG'), ('Everton', 'ORG')]
She was born in Leeds . -> [('Leeds', 'LOC')]
